In [2]:
!pip install streamlit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 88.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 80.0 MB/s eta 0:00:00


In [5]:
"""
Fake News Detection — Streamlit Web App
Run from terminal: streamlit run app.py
Make sure model.pkl and tfidf_vectorizer.pkl exist in the same folder.
"""

import streamlit as st
import pickle
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

# Download NLTK data quietly
nltk.download('stopwords', quiet=True)

# ─── PAGE CONFIGURATION ───────────────────────────────────────────────────────
st.set_page_config(
    page_title="Fake News Detector",
    page_icon="📰",
    layout="centered"
)

# ─── LOAD MODEL ───────────────────────────────────────────────────────────────
@st.cache_resource   # Cache so model loads only once
def load_model():
    with open('model.pkl', 'rb') as f:
        model = pickle.load(f)
    with open('tfidf_vectorizer.pkl', 'rb') as f:
        tfidf = pickle.load(f)
    return model, tfidf

try:
    model, tfidf = load_model()
    model_loaded = True
except FileNotFoundError:
    model_loaded = False

# ─── TEXT CLEANING (must match training exactly) ───────────────────────────────
stemmer = PorterStemmer()
stop_words = set(stopwords.words('english'))

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'http\S+|www\S+', '', text)
    text = re.sub(r'[^a-z\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    words = text.split()
    words = [stemmer.stem(w) for w in words if w not in stop_words and len(w) > 2]
    return ' '.join(words)

def predict(text):
    cleaned = clean_text(text)
    vectorized = tfidf.transform([cleaned])
    prediction = model.predict(vectorized)[0]
    probability = model.predict_proba(vectorized)[0]
    return prediction, probability

# ─── UI ───────────────────────────────────────────────────────────────────────
st.title("📰 Fake News Detector")
st.markdown("Paste any news article below to check if it's **Real** or **Fake**.")
st.markdown("---")

if not model_loaded:
    st.error("❌ Model files not found! Please run Step2_Preprocessing_and_Model.ipynb first to generate `model.pkl` and `tfidf_vectorizer.pkl`.")
    st.stop()

# Input area
news_input = st.text_area(
    label="📝 Enter News Article Text:",
    placeholder="Paste a news headline or full article here...",
    height=200
)

# Analyse button
if st.button("🔍 Analyse Article", use_container_width=True):
    if not news_input.strip():
        st.warning("⚠️ Please enter some text first!")
    elif len(news_input.strip().split()) < 5:
        st.warning("⚠️ Please enter at least a few sentences for a reliable prediction.")
    else:
        with st.spinner("Analysing..."):
            prediction, probability = predict(news_input)

        st.markdown("---")
        st.subheader("🔎 Result")

        # Large result display
        if prediction == 1:
            st.success("## ✅ REAL NEWS")
            result_color = "green"
        else:
            st.error("## 🚨 FAKE NEWS")
            result_color = "red"

        # Confidence scores
        col1, col2 = st.columns(2)
        with col1:
            st.metric(
                label="🚨 Fake Probability",
                value=f"{probability[0]*100:.1f}%"
            )
        with col2:
            st.metric(
                label="✅ Real Probability",
                value=f"{probability[1]*100:.1f}%"
            )

        # Progress bars
        st.markdown("**Confidence Breakdown:**")
        st.markdown("🚨 Fake News")
        st.progress(float(probability[0]))
        st.markdown("✅ Real News")
        st.progress(float(probability[1]))

        # Disclaimer
        st.markdown("---")
        st.caption("⚠️ This model is trained on a specific dataset and may not be accurate for all types of news. Always verify with trusted sources.")

# ─── SIDEBAR ─────────────────────────────────────────────────────────────────
with st.sidebar:
    st.header("ℹ️ About This App")
    st.markdown("""
    **Fake News Detection System**

    Built using:
    - Python
    - NLTK (text preprocessing)
    - Scikit-learn (TF-IDF + ML model)
    - Streamlit (web interface)

    **How it works:**
    1. Your text is cleaned and preprocessed
    2. Words are converted to TF-IDF numerical features
    3. A trained ML model classifies as Fake or Real
    4. Confidence probability is displayed
    """)

    st.markdown("---")
    st.header("📊 Model Info")
    if model_loaded:
        st.success("✅ Model loaded successfully")
        model_name = type(model).__name__
        st.write(f"**Model:** {model_name}")
        st.write(f"**Features:** {tfidf.max_features:,}")
    else:
        st.error("❌ Model not loaded")

    st.markdown("---")
    st.header("💡 Tips")
    st.markdown("""
    - Paste full articles for best accuracy
    - Works best with English news text
    - Minimum 5 words recommended
    """)

2026-03-25 09:34:55.681 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-25 09:34:55.685 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-25 09:34:55.766 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-25 09:34:55.768 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-25 09:34:55.768 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-25 09:34:55.770 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-25 09:34:55.772 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-25 09:34:55.773 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bar